In [1]:
import cv2
import mediapipe as mp
import numpy as np

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)

OpenCV: 5.0.0
MediaPipe: 0.10.9
NumPy: 2.4.6


In [2]:
mppose = mp.solutions.pose
mpdraw = mp.solutions.drawing_utils

In [3]:
data = mppose.Pose()

In [4]:
bowling_hand=input("Left or right")

In [5]:
elbow_angles=[]

In [6]:
import math

bowling_hand = bowling_hand.lower()

f=0
all_frame_data = []
elbow_angles = []
start_found = False
release_found=False
start_frame = None
start_angle = None


video = cv2.VideoCapture("videos/l2.mp4")
while True:
    suc, img = video.read()
    frame_data = []
    if not suc:
        break

    img1 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = data.process(img1)

    if result.pose_landmarks:
        f += 1
        landmarks = result.pose_landmarks.landmark

        if bowling_hand == "r":
            selected_landmarks = [
                mppose.PoseLandmark.RIGHT_SHOULDER,
                mppose.PoseLandmark.RIGHT_ELBOW,
                mppose.PoseLandmark.RIGHT_WRIST
            ]
        else:
            selected_landmarks = [
                mppose.PoseLandmark.LEFT_SHOULDER,
                mppose.PoseLandmark.LEFT_ELBOW,
                mppose.PoseLandmark.LEFT_WRIST
            ]

        for landmark_id in selected_landmarks:
            landmark = landmarks[landmark_id]
            frame_data.append([
                f,
                landmark_id,
                landmark.x,
                landmark.y,
                landmark.z,
                landmark.visibility
            ])

        
        all_frame_data.append(frame_data)



        mpdraw.draw_landmarks(
            img,
            result.pose_landmarks,
            mppose.POSE_CONNECTIONS
        )

        

    cv2.imshow("img", img)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()


In [7]:
best_horizontal=1
best_h_wrist=1

best_release=1
best_e_w=1

start_index = None
release_index = None

start_found=False

for index,frame in enumerate(all_frame_data):
    shoulder=frame[0]
    elbow=frame[1]
    wrist=frame[2]
    if elbow[2]<shoulder[2] and wrist[3]<shoulder[3]:
        if abs(shoulder[3]-elbow[3])<best_horizontal and abs(elbow[3]-wrist[3])<best_h_wrist:
            best_horizontal=abs(shoulder[3]-elbow[3])
            best_h_wrist=abs(elbow[3]-wrist[3])

            best_h_start=[shoulder[3],elbow[3],wrist[3]]

            best_start_frame=frame
            start_index=index
            start_found=True
            

if start_found is True:
    for i in range(start_index+1,len(all_frame_data)):
        frame=all_frame_data[i]
        shoulder=frame[0]
        elbow=frame[1]
        wrist=frame[2]
        if start_index is not None:
         if elbow[3]<shoulder[3]:
            if abs(shoulder[2]-elbow[2])<best_release and abs(elbow[2]-wrist[2])<best_e_w:
                best_release=abs(shoulder[2]-elbow[2])
                best_e_w=abs(elbow[2]-wrist[2])

                best_r=[shoulder[2],elbow[2],wrist[2]]

                best_release_frame=frame
                release_index=i
                
                


In [8]:
print(start_index)
print(release_index)

114
130


In [9]:
window=all_frame_data[start_index:release_index+1]

In [10]:
elbow_angles = []

import math

for f in window:

    shoulder = f[0]
    elbow = f[1]
    wrist = f[2]

    sx, sy = shoulder[2], shoulder[3]
    ex, ey = elbow[2], elbow[3]
    wx, wy = wrist[2], wrist[3]

    # Shoulder -> Elbow
    a = (
        sx - ex,
        sy - ey
    )

    # Wrist -> Elbow
    b = (
        wx - ex,
        wy - ey
    )

    dot = (
        a[0] * b[0] +
        a[1] * b[1]
    )

    mag_a = math.sqrt(
        a[0]**2 +
        a[1]**2
    )

    mag_b = math.sqrt(
        b[0]**2 +
        b[1]**2
    )

    if mag_a != 0 and mag_b != 0:

        value = dot / (mag_a * mag_b)

        # Prevent acos domain error
        value = max(-1, min(1, value))

        angle = math.degrees(
            math.acos(value)
        )

        elbow_angles.append(angle)

        print(
            "Frame:", f[0][0],
            "Angle:", angle
        )

Frame: 115 Angle: 169.00202619058345
Frame: 116 Angle: 171.8493292051372
Frame: 117 Angle: 170.36205732184277
Frame: 118 Angle: 169.05992018230913
Frame: 119 Angle: 177.705830400055
Frame: 120 Angle: 178.17824355044027
Frame: 121 Angle: 178.52728206048235
Frame: 122 Angle: 178.35273516561375
Frame: 123 Angle: 171.2220747516177
Frame: 124 Angle: 168.75385521490443
Frame: 125 Angle: 167.68647889329313
Frame: 126 Angle: 166.89057287598055
Frame: 127 Angle: 166.75579818245234
Frame: 128 Angle: 167.37498237004073
Frame: 129 Angle: 168.28678637037424
Frame: 130 Angle: 169.730387899148
Frame: 131 Angle: 175.19813602162475


In [11]:
print(elbow_angles)

[169.00202619058345, 171.8493292051372, 170.36205732184277, 169.05992018230913, 177.705830400055, 178.17824355044027, 178.52728206048235, 178.35273516561375, 171.2220747516177, 168.75385521490443, 167.68647889329313, 166.89057287598055, 166.75579818245234, 167.37498237004073, 168.28678637037424, 169.730387899148, 175.19813602162475]


In [12]:
print("MAX ANGLE:", max(elbow_angles))
print("MIN ANGLE:", min(elbow_angles))
print("EXTENSION:",max(elbow_angles)-min(elbow_angles))

MAX ANGLE: 178.52728206048235
MIN ANGLE: 166.75579818245234
EXTENSION: 11.771483878030011
